# Train a PPP policy: inspect the learning signal

This notebook is an **analysis walkthrough**, not a second trainer. It is self-contained and runs in a fresh Google Colab or local Jupyter kernel with only the Python standard library. The small functions below mirror the public `ppp_simplified.training` contracts so the teaching-scale RL signal is inspectable before a GPU instance exists.

Everything below is either **deterministic/sample** or loaded from a sanitized artifact. No cell contacts Gemini, loads Qwen, starts Lambda compute, or performs a policy update. A later live artifact can replace the sample with the same schema.

## 1. Paper-scale versus teaching-scale configuration

The important invariant is a group of eight trajectories. The small run reduces the number of prompt groups and training steps; it does **not** claim to reproduce the paper's performance.

In [ ]:
from dataclasses import asdict, dataclass
import hashlib
import json
import statistics

@dataclass(frozen=True)
class PPPTrainingConfig:
    """Notebook mirror of the reduced configuration; the trainer remains authoritative."""
    model_id: str = 'Qwen/Qwen3.5-4B'
    policy_version: str = 'navigation-v2'
    tool_schema_version: str = 'v2'
    group_size: int = 8
    prompt_groups_per_update: int = 1
    initial_steps: int = 20
    maximum_steps: int = 40
    max_turns: int = 8
    response_tokens: int = 4096
    lora_rank: int = 16
    lora_alpha: int = 32

    @property
    def identity(self):
        encoded = json.dumps(asdict(self), sort_keys=True).encode()
        return hashlib.sha256(encoded).hexdigest()

def group_advantages(rewards):
    """FoldGRPO normalization using sample standard deviation, like Verl."""
    if not rewards:
        raise ValueError('At least one reward is required.')
    if len(rewards) == 1:
        return (0.0,)
    mean = statistics.fmean(rewards)
    deviation = statistics.stdev(rewards)
    if deviation == 0:
        return tuple(0.0 for _ in rewards)
    return tuple((value - mean) / (deviation + 1e-6) for value in rewards)

def model_token_mask(turns):
    """Mark model tokens as 1 and environment tokens as 0."""
    mask = []
    for source, count in turns:
        if source not in {'model', 'environment'} or count < 0:
            raise ValueError(f'Invalid turn metadata: {(source, count)}')
        mask.extend([1 if source == 'model' else 0] * count)
    return tuple(mask)

def terminal_reward_vector(*, response_mask, reward):
    """Place terminal reward on the last model-generated token."""
    vector = [0.0] * len(response_mask)
    generated = [index for index, keep in enumerate(response_mask) if keep]
    if generated:
        vector[generated[-1]] = float(reward)
    return tuple(vector)

config = PPPTrainingConfig()
paper_vs_teaching = {
    'base model': ('Seed-OSS-36B', config.model_id),
    'trajectories per prompt': (8, config.group_size),
    'prompt groups per update': (64, config.prompt_groups_per_update),
    'training steps': (200, f'{config.initial_steps} initially; {config.maximum_steps} maximum'),
    'maximum logical turns': ('paper-scale run', config.max_turns),
    'response-token budget': (32000, config.response_tokens),
    'update': ('full-scale RL', f'BF16 LoRA r={config.lora_rank}, alpha={config.lora_alpha}'),
}
for setting, (paper, teaching) in paper_vs_teaching.items():
    print(f'{setting:28} paper={str(paper):24} teaching={teaching}')

print('\nFrozen policy/tool contract:', config.policy_version, config.tool_schema_version)
print('Notebook config fingerprint:', config.identity[:12], '...')

## 2. Load a sanitized artifact, with a sample fallback

Run the cell as-is to use the inline sample. To inspect a real sanitized export, either set `ARTIFACT_PATH` to a local/Google Drive path or set `UPLOAD_ARTIFACT_IN_COLAB = True` and rerun the cell to choose a JSON file from your computer. No repository checkout is required. The sample is deliberately labelled `deterministic_smoke`; it is not evidence of trained-model behavior.

In [ ]:
import os
from pathlib import Path

# Colab-friendly inputs. Leave both defaults unchanged for the built-in sample.
ARTIFACT_PATH = os.environ.get('PPP_TRAINING_ARTIFACT')
UPLOAD_ARTIFACT_IN_COLAB = False

SAMPLE_ARTIFACT = {
    'schema_version': 1,
    'stage': 'deterministic_smoke',
    'configuration': asdict(config),
    'dataset': {'selection_seed': 42, 'issue_groups': ['sample__issue-1']},
    'metrics': {'note': 'Inline deterministic sample; no model weights were updated.'},
    'reports': [{
        'instance_id': 'sample__issue-1',
        'repository': 'sample/repository',
        'visible_issue': 'A public API call fails for one edge case.',
        'is_vague': True,
        'preference': {'name': 'one_question', 'description': 'Ask one focused question.'},
        'model': 'scripted-smoke-agent',
        'simulator': 'deterministic',
        'predicted_functions': ['pkg/api.py:Client.send'],
        'reward': {'productivity': 1.0, 'proactivity_adjustment': 0.05, 'personalization_adjustment': 0.0, 'total': 1.0},
        'trajectory': [{
            'turn': 1, 'attempt': 1, 'tool': 'find_symbol',
            'arguments': {'name': 'send', 'path': 'pkg'},
            'reasoning': 'Locate the public request path.',
            'observation': '<repository observation: 32 characters redacted>',
            'executed': True, 'duplicate_suppressed': False,
        }, {
            'turn': 2, 'attempt': 1, 'tool': 'finish',
            'arguments': {'functions': ['pkg/api.py:Client.send']},
            'reasoning': 'This method is the localized target.',
            'observation': '<finish validation: 15 characters redacted>',
            'executed': True, 'duplicate_suppressed': False,
        }],
        'termination': 'natural_finish', 'model_calls': 2,
        'finish_validation_passed': True,
        'finish_correction_attempted': False,
        'invalid_predictions': [], 'inference_seed': 11,
    }],
}

if UPLOAD_ARTIFACT_IN_COLAB:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Please upload exactly one sanitized JSON artifact.')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    artifact = json.loads(uploaded_bytes.decode('utf-8'))
    artifact_source = f'uploaded artifact: {uploaded_name}'
else:
    artifact_path = Path(ARTIFACT_PATH).expanduser() if ARTIFACT_PATH else None
    if artifact_path and artifact_path.is_file():
        artifact = json.loads(artifact_path.read_text())
        artifact_source = f'artifact file: {artifact_path}'
    else:
        artifact = SAMPLE_ARTIFACT
        artifact_source = 'inline deterministic/sample fallback'

print('Artifact source:', artifact_source)
print('Stage:', artifact['stage'])
print('Live-trained?', artifact['stage'] == 'training')
print('Reports:', len(artifact['reports']))

## 3. One group of eight: terminal rewards become FoldGRPO advantages

Each of eight independently sampled trajectories for the same prompt receives one terminal reward. `group_advantages` uses sample-standard-deviation normalization. A flat group has no ranking signal and therefore produces all-zero advantages.

In [ ]:
# Deterministic teaching example: eight outcomes for one logical prompt.
group_rewards = (0.00, 0.15, 0.15, 0.40, 0.55, 0.55, 0.80, 1.00)
advantages = group_advantages(group_rewards)
for rollout, (reward, advantage) in enumerate(zip(group_rewards, advantages), start=1):
    print(f'rollout {rollout}: terminal_reward={reward:>4.2f}  advantage={advantage:>+6.3f}')

assert len(advantages) == config.group_size
assert group_advantages((0.5,) * config.group_size) == (0.0,) * config.group_size
print('\nA flat reward group has no within-group preference signal.')

## 4. Only model tokens get policy loss

Tool observations and user replies are environment tokens. They help the model choose its next action, but the policy loss must not train the model to reproduce them. The terminal reward is placed on the final generated token in the sequence.

In [ ]:
# Token counts are illustrative metadata, not real tokenizer output.
turns = (('model', 5), ('environment', 7), ('model', 4), ('environment', 3), ('model', 2))
response_mask = model_token_mask(turns)
reward = artifact['reports'][0]['reward']['total']
reward_vector = terminal_reward_vector(response_mask=response_mask, reward=reward)

print('model/environment mask:', response_mask)
print('trainable model tokens:', sum(response_mask), 'of', len(response_mask))
print('terminal reward vector:', reward_vector)
assert all(value == 0 for value, keep in zip(reward_vector, response_mask) if not keep)
assert reward_vector[-1] == reward

## 5. Inspect what a learner-visible trajectory contains

Artifacts are sanitized: they retain the visible issue, actions, observations, terminal prediction, and decomposed reward. They do not expose the full issue, patch, expected functions, base commit, workspace path, or API keys.

In [ ]:
report = artifact['reports'][0]
print('Issue:', report['visible_issue'])
print('Preference:', report['preference']['name'])
print('Prediction:', report['predicted_functions'])
print('Reward:', report['reward'])
print('Termination:', report['termination'])
for step in report['trajectory']:
    status = 'executed' if step['executed'] else 'suppressed'
    print(f"turn {step['turn']} attempt {step['attempt']} | {step['tool']} ({status})")
    print('  observation:', step['observation'])

assert 'full_issue' not in report
assert 'expected_functions' not in report

## 6. Before paid compute: reflect, then use the training CLI

Reflection prompts:

1. Which rollout would receive the largest positive update, and why?
2. How would a strong but overly chatty trajectory change the three reward components?
3. Why is an all-zero advantage group valid rather than a bug?
4. Which visible artifact field would let you distinguish a natural finish from an invalid finish correction?

The commands below match the current `ppp-train` CLI. They are shown for the learner to copy into a terminal; this notebook deliberately does **not** execute them. The first three are no-GPU preparation, while `--execute` is guarded because it begins billable GPU work.

```bash
# no-cost preparation and deterministic inspection
ppp-train doctor
ppp-train prepare --data data/train_12n1.parquet --selection-seed 42 --output-dir simplified/results/training/prepared
# Use a pinned checkout to avoid a repository fetch during the deterministic smoke group.
ppp-train smoke --data data/test_id.parquet --instance-id pallets__flask-5014 --workspace simplified/workspaces/pallets__flask-5014 --output simplified/results/training/deterministic-smoke.json

# A6000 compatibility gate: print the exact command first (safe, no GPU work).
ppp-train train --steps 1 --simulator deterministic --model qwen35
# After reviewing the printed command and explicitly accepting Lambda billing, execute one step.
CONFIRM_PAID_TRAINING=I_UNDERSTAND_LAMBDA_IS_BILLING ppp-train train --steps 1 --simulator deterministic --model qwen35 --execute

# If the saved Qwen3.5 compatibility evidence warrants the documented fallback, print/run the same guarded one-step gate.
ppp-train train --steps 1 --simulator deterministic --model qwen3

# One real eight-trajectory Qwen/Gemini group (requires GEMINI_API_KEY; no optimizer).
ppp-train live-group --data data/test_id.parquet --instance-id pallets__flask-5014 --workspace simplified/workspaces/pallets__flask-5014 --model qwen35 --inference-seeds 11,22,33,44,55,66,77,88 --output simplified/results/training/live-group.json

# Only after the one-step gate and live group are inspected: a guarded 20-step teaching run.
CONFIRM_PAID_TRAINING=I_UNDERSTAND_LAMBDA_IS_BILLING ppp-train train --steps 20 --simulator gemini --model qwen35 --execute
# Rerun the 20-step command once to verify resume, then extend only if the automated gate passes.
CONFIRM_PAID_TRAINING=I_UNDERSTAND_LAMBDA_IS_BILLING ppp-train train --steps 40 --simulator gemini --model qwen35 --projected-compute-usd 44 --execute

# Evaluation prints the paired vLLM-serving and frozen dev-v1 evaluation commands.
ppp-train evaluate --adapter simplified/results/checkpoints/qwen35/<adapter-dir> --model qwen35 --output-dir simplified/results/training/adapter-dev-v1
# Export validates the sanitized JSON and writes a learner-readable summary.
ppp-train export --artifact simplified/results/training/live-group.json --output artifacts/training.json --summary artifacts/training.md
```

When a real exported artifact is available, rerun this notebook with `PPP_TRAINING_ARTIFACT=artifacts/training.json`. Treat a 20-step result as a methodology demonstration, not a performance reproduction.